In [7]:
from trading_engine.core import (
    read_data, create_model_state, orchestrate_model_backtests,
    orchestrate_model_simulations, orchestrate_portfolio_optimizations,
    orchestrate_portfolio_simulations, orchestrate_portfolio_aggregation
)

import datetime

In [8]:
# 1) experiment config
universe = [
  "SPY-US", "SLV-US", "GLD-US", "TLT-US", "USO-US", "UNG-US", "IXJ-US",
  "KXI-US", "JXI-US", "IXG-US", "IXN-US", "RXI-US", "MXI-US", "EXI-US",
  "IXC-US", "IEI-US", "SHY-US", "BIL-US", "JPXN-US", "INDA-US", "MCHI-US",
  "EZU-US", "IBIT-US", "ETHA-US", "VIXY-US"
]
features = ["close_momentum_10"]                   # keys in FEATURES
models   = ["RXI_TLT_pml_10", "GLD_USO_nml_10"]    # keys in MODELS
aggregators = ["model_mvo"]                        # keys in AGGREGATORS
optimizers   = ["mean_variance_constrained"]       # keys in OPTIMIZERS
initial_value = 1_000_000
start_date = datetime.date(2021, 1, 1)
end_date = datetime.date(2025, 1, 1)

In [9]:
# 2) build model state + prices (cached locally)
lf = read_data()
model_state, prices = create_model_state(
    lf=lf,
    features=features,
    start_date=start_date,
    end_date=end_date,
    universe=universe
)

In [10]:
# 3) run model backtests + simulations
model_insights = orchestrate_model_backtests(
    model_state=model_state,
    models=models,
    universe=universe
)

model_simulations = orchestrate_model_simulations(
    prices=prices,
    model_insights=model_insights,
    start_date=start_date,
    end_date=end_date
)

In [11]:
# 4) aggregate + optimize portfolio and simulate
aggregated_insights = orchestrate_portfolio_aggregation(
    model_insights=model_insights,
    backtest_results=model_simulations,
    universe=universe,
    aggregators=aggregators,
    start_date=start_date,
    end_date=end_date,
)

optimizer_insights = orchestrate_portfolio_optimizations(
    prices=prices,
    aggregated_insights=aggregated_insights,
    universe=universe,
    optimizers=optimizers,
)

optimizer_simulations = orchestrate_portfolio_simulations(
    prices=prices,
    portfolio_insights=optimizer_insights,
    start_date=start_date,
    end_date=end_date,
    initial_value=initial_value,
)

In [6]:
# 5) visualize one result (example: mean_variance_constrained)
optimizer_simulations["mean_variance_constrained"]["backtest_metrics"]

metric,value
str,f64
"""total_return""",0.421066
"""annualized_return""",0.092113
"""annualized_volatility""",0.07742
"""sharpe_ratio""",1.189776
"""sortino_ratio""",1.04607
…,…
"""num_weight_events""",1005.0
"""parsing_time_ms""",17.0
"""simulation_time_ms""",21.0
